In [ ]:
# Liste öffnen


import json
import pprint

file_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\ds4glam\data\authorities-gnd-koerperschaft_lds_20250313.jsonld"

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

count = 0
max_items = 100

for inner_list in data:
    if not isinstance(inner_list, list):
        continue  # just in case, skip if not a list
    for item in inner_list:
        pprint.pprint(item)
        print('-' * 40)
        count += 1
        if count >= max_items:
            break
    if count >= max_items:
        break

In [ ]:
# Liste Hauptansetzungen

import json
import csv

file_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\ds4glam\data\authorities-gnd-koerperschaft_lds_20250313.jsonld"
output_csv = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\preferred_names.csv"

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# The key to look for
target_key = 'https://d-nb.info/standards/elementset/gnd#preferredNameForTheCorporateBody'

preferred_names = []

for inner_list in data:
    if not isinstance(inner_list, list):
        continue
    for entry in inner_list:
        # Check if target key exists
        if target_key in entry:
            # entry[target_key] is a list of dicts with '@value'
            for val_dict in entry[target_key]:
                if '@value' in val_dict:
                    preferred_names.append(val_dict['@value'])

# Write to CSV
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['preferredNameForTheCorporateBody'])  # header
    for name in preferred_names:
        writer.writerow([name])

print(f"Extracted {len(preferred_names)} preferred names and saved to {output_csv}")

In [ ]:
# Listen vergleichen und Dubletten finden

import pandas as pd

# --- Step 1: Load preferred names from CSV ---
csv_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\preferred_names.csv"
preferred_df = pd.read_csv(csv_path)

# Count occurrences of each preferred name
preferred_counts = preferred_df['preferredNameForTheCorporateBody'].dropna().astype(str).value_counts()
preferred_2plus = set(preferred_counts[preferred_counts >= 2].index)

# --- Step 2: Load 'title' column from Excel ---
excel_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\ds4glam\data\VERZ_b-Level6-BaUGNF-V_20250326_nme-emo.xlsx"
excel_df = pd.read_excel(excel_path)

if 'title' not in excel_df.columns:
    raise ValueError("Column 'title' not found in Excel file.")

# Get unique titles from Excel
excel_titles = set(excel_df['title'].dropna().astype(str))

# --- Step 3: Find names in both datasets with the required conditions ---
common_filtered = list(preferred_2plus & excel_titles)

# --- Step 4: Save results ---
output_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\common_titles_2plus_csv_1plus_excel.csv"
pd.DataFrame({'common_titles': common_filtered}).to_csv(output_path, index=False)

print(f"✅ Found {len(common_filtered)} names that appear ≥2 times in CSV and ≥1 time in Excel.")
print(f"📁 Saved to: {output_path}")

In [ ]:
# Mit Wikipedia abgleichen

import pandas as pd
import wikipedia
import time

# --- Load the list of common titles ---
input_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\common_titles_2plus_csv_1plus_excel.csv"
df = pd.read_csv(input_path)

# --- Limit to the first 50 entries ---
names = df['common_titles'].dropna().astype(str).tolist()[:50]

# --- Check Wikipedia for matching pages ---
results = []

for name in names:
    try:
        page = wikipedia.page(name, auto_suggest=False, redirect=True)
        results.append({'name': name, 'wikipedia_url': page.url})
    except (wikipedia.exceptions.DisambiguationError, wikipedia.exceptions.PageError):
        pass  # Skip names with no clear Wikipedia page
    time.sleep(0.5)  # Be kind to Wikipedia servers

# --- Save only positives (with valid pages) ---
results_df = pd.DataFrame(results)

output_path = r"C:\Users\scaale00\Documents\DS4Glam\ds4glam\wikipedia_found_top50.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ {len(results_df)} entries have Wikipedia pages.")
print(f"📁 Saved to: {output_path}")